# Google Colab - Remote Ollama

A Google Colab notebook that runs [Ollama](https://ollama.com/) and exposes it to the internet, enabling you to do the heavy lifting in the cloud and access it from your local machine .

### Get Started
1. Open the notebook in Google Colab
2. Run each cell sequentially:
   - Cell 1: Installs Ollama and dependencies
   - Cell 2: Starts the Ollama server in the background
   - Cell 3: Creates a public tunnel and outputs a URL for accessing Ollama
3. Use the generated URL to connect to Ollama from any external application
4. Once the tunnel is active, you can use Ollama like this:
   ```bash
   export OLLAMA_HOST=http://<generated-url>
   ollama run granite4:3b --prompt "What is the capital of France?"
   ollama ls
   ollama ps
   ``` 


### Configuration
- **Port**: `11434` (exposed on all interfaces)
- **Context Length**: 128,000 tokens

# Get Started

In [ ]:
# Install Ollama & nessesary Linux Packages
!sudo apt-get update; sudo apt-get install zstd tmux ;  curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Start Ollama With 128k Token in background
!tmux new -d -s ollama 'OLLAMA_HOST=0.0.0.0:11434 OLLAMA_CONTEXT_LENGTH=128000 ollama serve'

In [ ]:
import pexpect
import re

cmd = "ssh -p 443 -o StrictHostKeyChecking=no -R0:localhost:11434 qr@free.pinggy.io"

child = pexpect.spawn(cmd, encoding="utf-8")

url = None

while True:
    try:
        i = child.expect([
            r"password:",
            r"https?://[^\s\x1b]+",
            pexpect.EOF,
            pexpect.TIMEOUT
        ], timeout=5)

        if i == 0:
            child.sendline("7530")

        elif i == 1:
            raw = child.match.group(0)
            url = re.sub(r'\x1b\[[0-9;]*[A-Za-z]', '', raw).strip()
            print(url)
            continue

        elif i == 2:
            continue

    except pexpect.TIMEOUT:
        continue

child.close()


### Models you can run with collab T4-GPU (16GB VRAM limit):
1. `granite4:3b` (3 billion parameters)
2. `gpt-oss:20b` (20 billion parameters)
3. `qwen3.5:9b` (9 billion parameters)
4. same alike.....